# The Harvard Pipeline — Somali Traditional Music, End to End

**Somali Music AI Preservation Platform · Phase 3 (AI research pipeline)**
Khalid Ibrahim — Somali-American AI engineer, Minneapolis MN

This notebook processes **605 Somali traditional music recordings** digitised from
Harvard University's **Archive of World Music** into a research-grade, annotated
dataset: cleaned audio, source-separated stems, Somali + English transcripts,
frame-level pitch tracks mapped onto the **Somali pentatonic scale**, and the
per-note **microtonal deviation from Western equal temperament (in cents)** that is
the central research contribution of the accompanying ISMIR paper
(`docs/ISMIR_DRAFT.md` in the platform repository).

### Why this matters

Somali music — *heello*, *qaraami*, *dhaanto*, *buraanbur* — is a living oral
tradition with **no representation in any major MIR corpus**. Its pitch
organisation does not align with twelve-tone equal temperament (12-TET), and no
Western-trained audio model has ever measured *how* it deviates. Step 7 of this
pipeline quantifies exactly that, frame by frame, across the corpus.

### Pipeline overview

```
02_converted_wav/  (605 WAV, Harvard AWM)
   │
   ├─ STEP 1  inventory + total duration
   ├─ STEP 2  filename → metadata          → metadata_raw.csv
   ├─ STEP 3  quality audit (SNR gate)     → quality_report.csv
   ├─ STEP 4  DeepFilterNet denoise        → 03_cleaned_wav/
   ├─ STEP 5  Demucs source separation     → 04_separated/track_XXXX/{vocals,no_vocals}.wav
   ├─ STEP 6  Whisper large-v3 (so + en)   → metadata_transcripts.json
   ├─ STEP 7  CREPE f0 → Somali scale map  → pitch_data/track_XXXX_pitch.json
   ├─ STEP 8  assemble dataset             → somali_music_dataset_v1.{json,csv}
   └─ STEP 9  statistics + Figure 1        → dataset_summary.txt, figure1_cents_histograms.png
```

### Design commitments

1. **Resumable everywhere.** Colab disconnects; every per-file step checks for its
   output before recomputing, and every step writes its results to Google Drive as
   it goes. Re-running the notebook top to bottom after a disconnect only does the
   remaining work.
2. **Originals are never modified.** `02_converted_wav/` is read-only input; every
   derivative goes to its own directory. Provenance is the dataset's value.
3. **The scale mapping is the platform's canonical code.** The functions in Step 7
   are copied **verbatim** from `apps/ai-service/services/scale.py` — the same
   unit-tested module the production AI service runs — so the numbers reported here
   are exactly reproducible by the platform and the paper.
4. **No fabricated labels.** Cultural fields the pipeline cannot honestly infer
   (genre, era) are carried as *candidates for expert review*, never invented.

### Runtime

Use a **GPU runtime** (T4 is enough; A100 is faster). Steps 4–7 are the heavy ones
— expect several hours end-to-end for 605 tracks on a T4. The notebook may be run
step by step across multiple sessions; that is the intended workflow.


## Configuration

One place for every path and constant. `CONFIDENCE_THRESHOLD` (τ = 0.80) matches
the platform default (`apps/ai-service/config.py: pitch_confidence_threshold`), so
notebook results and production results gate pitch frames identically.


In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path

# ── Google Drive layout ──────────────────────────────────────────────────────
DRIVE_ROOT = Path("/content/drive/MyDrive/QaraamiGen")
RAW_WAV_DIR = DRIVE_ROOT / "02_converted_wav"     # input — never written to
CLEANED_DIR = DRIVE_ROOT / "03_cleaned_wav"       # Step 4 output
SEPARATED_DIR = DRIVE_ROOT / "04_separated"       # Step 5 output
PITCH_DIR = DRIVE_ROOT / "pitch_data"             # Step 7 output
REPORTS_DIR = DRIVE_ROOT / "00_reports"           # CSV/JSON reports, figures

METADATA_RAW_CSV = REPORTS_DIR / "metadata_raw.csv"
QUALITY_CSV = REPORTS_DIR / "quality_report.csv"
TRANSCRIPTS_JSON = REPORTS_DIR / "metadata_transcripts.json"
DATASET_JSON = REPORTS_DIR / "somali_music_dataset_v1.json"
DATASET_CSV = REPORTS_DIR / "somali_music_dataset_v1.csv"
SUMMARY_TXT = REPORTS_DIR / "dataset_summary.txt"
FIGURE1_PNG = REPORTS_DIR / "figure1_cents_histograms.png"

# ── Analysis constants ───────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.80   # τ — CREPE confidence gate (platform default)
SNR_FLAG_DB = 20.0            # Step 3 quality gate
WHISPER_MODEL = "large-v3"
CREPE_STEP_MS = 10            # 10 ms frames (ARCHITECTURE.md §10)

# Whisper hallucination heuristics — mirror
# apps/ai-service/services/transcription_service.py exactly.
SUSPECT_COMPRESSION_RATIO = 2.4
SUSPECT_AVG_LOGPROB = -1.0
NO_SPEECH_PROB = 0.6


def ensure_dirs() -> None:
    for d in (CLEANED_DIR, SEPARATED_DIR, PITCH_DIR, REPORTS_DIR):
        d.mkdir(parents=True, exist_ok=True)


def save_json(path: Path, payload) -> None:
    """Atomic-ish JSON write: tmp file then rename, so a disconnect mid-write
    never corrupts a report we would later resume from."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
    tmp.replace(path)


def load_json(path: Path, default):
    if path.exists():
        return json.loads(path.read_text())
    return default


def gpu_report() -> None:
    try:
        import torch
        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
        else:
            print("WARNING: no GPU — steps 4-7 will be extremely slow. "
                  "Runtime → Change runtime type → GPU.")
    except ImportError:
        print("torch not installed yet (fine before Step 4).")


gpu_report()


## STEP 1 — Mount Google Drive and verify the corpus

Before any processing: confirm the 605 files are actually there, list a sample so
a human can eyeball the naming convention, and measure the total duration.
Duration is read from WAV headers (`soundfile.info`) — no audio is decoded, so
this scan is fast even over Drive.

The inventory is saved to `00_reports/file_inventory.csv` and reused by later
steps, so the Drive walk happens once.


In [ ]:
%pip -q install soundfile pandas tqdm

from google.colab import drive  # noqa: E402
drive.mount("/content/drive", force_remount=False)

import pandas as pd  # noqa: E402
import soundfile as sf  # noqa: E402
from tqdm.auto import tqdm  # noqa: E402

ensure_dirs()
assert RAW_WAV_DIR.exists(), f"Input directory missing: {RAW_WAV_DIR}"

INVENTORY_CSV = REPORTS_DIR / "file_inventory.csv"

wav_files = sorted(RAW_WAV_DIR.glob("*.wav"))
print(f"Total WAV files: {len(wav_files)}")
print("\nFirst 10 filenames:")
for f in wav_files[:10]:
    print("  ", f.name)

if INVENTORY_CSV.exists():
    inventory = pd.read_csv(INVENTORY_CSV)
    known = set(inventory["filename"])
    missing = [f for f in wav_files if f.name not in known]
    print(f"\nResuming inventory: {len(known)} known, {len(missing)} to scan.")
else:
    inventory = pd.DataFrame(columns=["filename", "duration_sec", "sample_rate", "channels"])
    missing = wav_files

rows = []
for f in tqdm(missing, desc="Scanning WAV headers"):
    try:
        info = sf.info(str(f))
        rows.append(
            {
                "filename": f.name,
                "duration_sec": round(info.duration, 2),
                "sample_rate": info.samplerate,
                "channels": info.channels,
            }
        )
    except Exception as exc:  # unreadable file — record it, keep going
        rows.append({"filename": f.name, "duration_sec": None,
                     "sample_rate": None, "channels": None})
        print(f"  UNREADABLE: {f.name} — {exc}")

if rows:
    inventory = pd.concat([inventory, pd.DataFrame(rows)], ignore_index=True)
    inventory.to_csv(INVENTORY_CSV, index=False)

total_sec = inventory["duration_sec"].fillna(0).sum()
hours, rem = divmod(int(total_sec), 3600)
minutes = rem // 60
print(f"\nTotal corpus duration: {hours}h {minutes}m "
      f"({total_sec:,.0f} seconds across {len(inventory)} files)")


## STEP 2 — Parse metadata from filenames

Files follow `track_XXXX_SongTitle_Artist.wav`. Underscores separate the fields
**and** appear inside multi-word titles/artist names, so pure filename parsing is
ambiguous. The strategy, in order of trust:

1. **Authoritative CSV.** If a metadata CSV exists anywhere in `QaraamiGen/`
   (the archive shipped one), it is located automatically and merged on
   `track_id` — its `title`/`artist` values override the filename parse.
2. **Filename heuristic.** Otherwise the last underscore token is taken as the
   artist and everything between the track number and it as the title, and the
   row is flagged `needs_review=True` whenever the split is ambiguous.

Nothing is guessed silently: every row records *which* source its metadata came
from. The output is `metadata_raw.csv` — "raw" because titles and artists await
verification against the Archive of World Music catalogue.


In [ ]:
import pandas as pd

FILENAME_RE = re.compile(r"^track_(\d+)_(.+)\.wav$", re.IGNORECASE)


def parse_filename(name: str) -> dict:
    """Best-effort parse of track_XXXX_Title_Artist.wav (documented heuristic)."""
    m = FILENAME_RE.match(name)
    if not m:
        return {"track_id": None, "title": None, "artists": None, "needs_review": True}
    track_id = int(m.group(1))
    rest = m.group(2).split("_")
    if len(rest) == 1:  # no artist segment at all
        return {"track_id": track_id, "title": rest[0].strip(),
                "artists": None, "needs_review": True}
    title = " ".join(rest[:-1]).strip()
    artists = rest[-1].strip()
    # Ambiguous whenever the artist could plausibly be more than one token.
    return {"track_id": track_id, "title": title, "artists": artists,
            "needs_review": len(rest) > 2}


inventory = pd.read_csv(REPORTS_DIR / "file_inventory.csv")
parsed = pd.DataFrame([{"filename": fn, **parse_filename(fn)} for fn in inventory["filename"]])
metadata = parsed.merge(inventory[["filename", "duration_sec"]], on="filename", how="left")
metadata["metadata_source"] = "filename"

# ── Look for the authoritative metadata CSV shipped with the archive ─────────
candidates = [
    p for p in DRIVE_ROOT.glob("*.csv")
    if p.parent == DRIVE_ROOT  # top level only — our own reports live in 00_reports/
]
print("Candidate metadata CSVs found:", [p.name for p in candidates] or "none")

for candidate in candidates:
    try:
        official = pd.read_csv(candidate)
    except Exception as exc:
        print(f"  skipping {candidate.name}: {exc}")
        continue
    cols = {c.lower().strip(): c for c in official.columns}
    id_col = next((cols[k] for k in ("track_id", "track", "id", "track_number") if k in cols), None)
    if id_col is None:
        print(f"  {candidate.name}: no track-id column recognised — skipped")
        continue
    official["_tid"] = pd.to_numeric(official[id_col], errors="coerce")
    title_col = next((cols[k] for k in ("title", "song_title", "song") if k in cols), None)
    artist_col = next((cols[k] for k in ("artists", "artist", "performer", "performers") if k in cols), None)
    merged = 0
    for _, row in official.dropna(subset=["_tid"]).iterrows():
        mask = metadata["track_id"] == int(row["_tid"])
        if not mask.any():
            continue
        if title_col and isinstance(row[title_col], str) and row[title_col].strip():
            metadata.loc[mask, "title"] = row[title_col].strip()
        if artist_col and isinstance(row[artist_col], str) and row[artist_col].strip():
            metadata.loc[mask, "artists"] = row[artist_col].strip()
        metadata.loc[mask, "metadata_source"] = f"csv:{candidate.name}"
        metadata.loc[mask, "needs_review"] = False
        merged += mask.sum()
    print(f"  {candidate.name}: enriched {merged} rows")

metadata = metadata[["track_id", "filename", "title", "artists",
                     "duration_sec", "metadata_source", "needs_review"]]
metadata = metadata.sort_values("track_id").reset_index(drop=True)
metadata.to_csv(METADATA_RAW_CSV, index=False)

print(f"\nSaved {METADATA_RAW_CSV.name}: {len(metadata)} rows, "
      f"{int(metadata['needs_review'].sum())} flagged for manual review")
metadata.head(10)


## STEP 3 — Audio quality audit

Archive digitisations vary: some transfers are clean, some carry tape hiss,
turntable rumble, or clipping. Before spending GPU-hours we measure every file:

- **sample rate / channels / duration** — from the header;
- **peak level (dBFS)** — clipping indicator;
- **SNR estimate (dB)** — computed from short-frame RMS energies: the noise floor
  is taken as the 10th percentile of frame RMS (the quietest passages, i.e. the
  medium's hiss between phrases) and the signal level as the 90th percentile.
  This is an *estimate* suited to ranking and gating, not a laboratory
  measurement, and it is documented as such in the dataset.

Files with `SNR < 20 dB` are **flagged, not dropped** — a historically unique
recording with tape hiss is still a historically unique recording. The flag feeds
the `quality_score` in Step 8 and tells DeepFilterNet's evaluation (Step 4) where
to look. Resumable: already-measured files are skipped on re-run.


In [ ]:
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

FRAME = 2048
HOP = 1024
EPS = 1e-10


def quality_metrics(path: Path) -> dict:
    audio, sr = sf.read(str(path), always_2d=True)
    mono = audio.mean(axis=1)
    peak = float(np.max(np.abs(mono))) if mono.size else 0.0
    peak_db = 20.0 * math.log10(max(peak, EPS))
    n = max(1, (len(mono) - FRAME) // HOP)
    rms = np.array([
        float(np.sqrt(np.mean(mono[i * HOP: i * HOP + FRAME] ** 2)))
        for i in range(n)
    ])
    noise_floor = max(float(np.percentile(rms, 10)), EPS)
    signal_level = max(float(np.percentile(rms, 90)), EPS)
    snr_db = 20.0 * math.log10(signal_level / noise_floor)
    return {
        "sample_rate": sr,
        "channels": audio.shape[1],
        "duration_sec": round(len(mono) / sr, 2),
        "peak_db": round(peak_db, 2),
        "snr_db": round(snr_db, 2),
    }


done: dict[str, dict] = {}
if QUALITY_CSV.exists():
    done = {r["filename"]: r for r in pd.read_csv(QUALITY_CSV).to_dict("records")}
    print(f"Resuming quality audit: {len(done)} files already measured.")

wav_files = sorted(RAW_WAV_DIR.glob("*.wav"))
errors = []
for f in tqdm(wav_files, desc="Measuring quality"):
    if f.name in done:
        continue
    try:
        row = {"filename": f.name, **quality_metrics(f)}
        row["flagged_low_snr"] = row["snr_db"] < SNR_FLAG_DB
        done[f.name] = row
    except Exception as exc:
        errors.append({"filename": f.name, "error": str(exc)})
    if len(done) % 25 == 0:  # checkpoint to Drive as we go
        pd.DataFrame(done.values()).to_csv(QUALITY_CSV, index=False)

quality = pd.DataFrame(done.values())
quality.to_csv(QUALITY_CSV, index=False)
if errors:
    pd.DataFrame(errors).to_csv(REPORTS_DIR / "errors_step3.csv", index=False)

flagged = quality[quality["flagged_low_snr"]]
print(f"\nMeasured {len(quality)} files — {len(flagged)} flagged below "
      f"{SNR_FLAG_DB:.0f} dB SNR ({len(errors)} unreadable).")
quality[["snr_db", "peak_db", "duration_sec"]].describe().round(2)


## STEP 4 — Noise removal (DeepFilterNet)

Archive transfers carry broadband tape hiss that degrades everything downstream —
source separation bleeds, Whisper mistranscribes, CREPE's confidence drops. We run
every file through **DeepFilterNet** (a real-time deep filtering network operating
at 48 kHz), writing results to `03_cleaned_wav/` and **never touching the
originals** — future researchers may prefer to re-clean from source with better
tools, and the raw transfer is itself part of the provenance record.

Resumable: a file whose cleaned counterpart already exists is skipped, so a Colab
disconnect costs only the file that was in flight.


In [ ]:
%pip -q install deepfilternet

import torch
from df.enhance import enhance, init_df, load_audio, save_audio
from tqdm.auto import tqdm

model, df_state, _ = init_df()  # downloads pretrained DeepFilterNet weights once
print(f"DeepFilterNet ready (model sr = {df_state.sr()} Hz)")

wav_files = sorted(RAW_WAV_DIR.glob("*.wav"))
todo = [f for f in wav_files if not (CLEANED_DIR / f.name).exists()]
print(f"{len(wav_files) - len(todo)} already cleaned, {len(todo)} to go.")

errors = []
for f in tqdm(todo, desc="DeepFilterNet"):
    try:
        audio, _ = load_audio(str(f), sr=df_state.sr())
        with torch.no_grad():
            cleaned = enhance(model, df_state, audio)
        save_audio(str(CLEANED_DIR / f.name), cleaned, df_state.sr())
    except Exception as exc:
        errors.append({"filename": f.name, "error": str(exc)})

if errors:
    import pandas as pd
    pd.DataFrame(errors).to_csv(REPORTS_DIR / "errors_step4.csv", index=False)
print(f"Done. Cleaned files: {len(list(CLEANED_DIR.glob('*.wav')))} "
      f"({len(errors)} failures logged)")


## STEP 5 — Source separation (Demucs)

Each cleaned recording is split into two stems with **Demucs (htdemucs)**:

- `vocals.wav` — the singer's voice → input to Whisper (Step 6);
- `no_vocals.wav` — oud and other instruments → input to CREPE (Step 7).

This separation is what lets the pitch analysis speak about the **instrumental
melodic line** without the voice confounding the f0 track, and lets Whisper hear
the voice without the oud pulling it towards hallucination. Layout:
`04_separated/track_XXXX/{vocals,no_vocals}.wav`. Resumable per track.


In [ ]:
%pip -q install demucs

import torch
from demucs.api import Separator, save_audio as demucs_save
from tqdm.auto import tqdm

separator = Separator(model="htdemucs", progress=False,
                      device="cuda" if torch.cuda.is_available() else "cpu")
print(f"Demucs htdemucs ready on {separator.device} (sr = {separator.samplerate})")

TRACK_ID_RE = re.compile(r"^track_(\d+)_", re.IGNORECASE)
cleaned_files = sorted(CLEANED_DIR.glob("*.wav"))
errors = []

for f in tqdm(cleaned_files, desc="Demucs"):
    m = TRACK_ID_RE.match(f.name)
    track_dir = SEPARATED_DIR / (f"track_{int(m.group(1)):04d}" if m else f.stem)
    vocals_out = track_dir / "vocals.wav"
    novocals_out = track_dir / "no_vocals.wav"
    if vocals_out.exists() and novocals_out.exists():
        continue
    try:
        _origin, stems = separator.separate_audio_file(str(f))
        vocals = stems["vocals"]
        no_vocals = sum(v for k, v in stems.items() if k != "vocals")
        track_dir.mkdir(parents=True, exist_ok=True)
        demucs_save(vocals, str(vocals_out), samplerate=separator.samplerate)
        demucs_save(no_vocals, str(novocals_out), samplerate=separator.samplerate)
    except Exception as exc:
        errors.append({"filename": f.name, "error": str(exc)})

if errors:
    import pandas as pd
    pd.DataFrame(errors).to_csv(REPORTS_DIR / "errors_step5.csv", index=False)
print(f"Done. Separated tracks: {len(list(SEPARATED_DIR.glob('track_*')))} "
      f"({len(errors)} failures logged)")


## STEP 6 — Transcription (Whisper large-v3, Somali)

Whisper large-v3 transcribes each `vocals.wav` twice: once with `language="so"`
(Somali text) and once with `task="translate"` (English). Somali is severely
under-represented in Whisper's training data **and sung text is not speech** —
Whisper is known to *hallucinate* fluent text over melismatic singing. An archive
that silently stored those hallucinations would poison every downstream use.

So every segment is scored with the same heuristics the production service uses
(`apps/ai-service/services/transcription_service.py`, thresholds identical):

| Signal | Threshold | Meaning |
|---|---|---|
| `compression_ratio` | > 2.4 | hallucination loops repeat text, compressing well |
| `avg_logprob` | < −1.0 | token confidence collapses on sung content |
| `no_speech_prob` | > 0.6 (majority vote) | Whisper's own "this isn't speech" detector |

A track whose segments majority-vote "not speech" — or where most segments look
suspect — is flagged `is_singing: true`, and its transcript is stored as
**advisory, not ground truth**. Per-segment confidence is `exp(avg_logprob)` (the
geometric-mean token probability). Progress is checkpointed to
`metadata_transcripts.json` after every track.


In [ ]:
%pip -q install openai-whisper

import torch
import whisper
from tqdm.auto import tqdm

model = whisper.load_model(WHISPER_MODEL)
use_fp16 = torch.cuda.is_available()
print(f"Whisper {WHISPER_MODEL} loaded (fp16={use_fp16})")


def segment_confidence(avg_logprob) -> float:
    if avg_logprob is None:
        return 0.0
    return max(0.0, min(1.0, math.exp(avg_logprob)))


def annotate(result: dict) -> dict:
    segments = result.get("segments", [])
    suspects = 0
    no_speech_votes = 0
    seg_rows = []
    for seg in segments:
        suspect = (
            seg.get("compression_ratio", 0.0) > SUSPECT_COMPRESSION_RATIO
            or seg.get("avg_logprob", 0.0) < SUSPECT_AVG_LOGPROB
        )
        no_speech = seg.get("no_speech_prob", 0.0) > NO_SPEECH_PROB
        suspects += suspect
        no_speech_votes += no_speech
        seg_rows.append(
            {
                "start": round(seg.get("start", 0.0), 2),
                "end": round(seg.get("end", 0.0), 2),
                "text": seg.get("text", "").strip(),
                "confidence": round(segment_confidence(seg.get("avg_logprob")), 3),
                "suspect_hallucination": bool(suspect),
            }
        )
    n = max(1, len(segments))
    return {
        "text": result.get("text", "").strip(),
        "segments": seg_rows,
        "is_singing": (no_speech_votes > n / 2) or (suspects > n / 2),
        "mean_confidence": round(sum(s["confidence"] for s in seg_rows) / n, 3),
    }


transcripts = load_json(TRANSCRIPTS_JSON, {})
print(f"Resuming transcription: {len(transcripts)} tracks already done.")

track_dirs = sorted(SEPARATED_DIR.glob("track_*"))
for track_dir in tqdm(track_dirs, desc="Whisper"):
    track_key = track_dir.name
    vocals = track_dir / "vocals.wav"
    if track_key in transcripts or not vocals.exists():
        continue
    try:
        so = model.transcribe(str(vocals), language="so", task="transcribe", fp16=use_fp16)
        en = model.transcribe(str(vocals), language="so", task="translate", fp16=use_fp16)
        so_ann, en_ann = annotate(so), annotate(en)
        transcripts[track_key] = {
            "somali": so_ann,
            "english": en_ann,
            "is_singing": so_ann["is_singing"],
            "whisper_model": WHISPER_MODEL,
        }
    except Exception as exc:
        transcripts[track_key] = {"error": str(exc)}
    save_json(TRANSCRIPTS_JSON, transcripts)  # checkpoint after every track

ok = sum(1 for v in transcripts.values() if "error" not in v)
singing = sum(1 for v in transcripts.values() if v.get("is_singing"))
print(f"\nTranscribed {ok}/{len(track_dirs)} tracks; "
      f"{singing} flagged is_singing (transcript advisory).")


## STEP 7 — Pitch extraction and the Somali pentatonic scale map

**This is the core research contribution.** CREPE extracts a frame-level
fundamental-frequency track from each `no_vocals.wav` (the oud line), and each
confident frame is mapped onto a five-degree Somali pentatonic reference:

| degree | reference | note |
|---|---|---|
| do | 293.66 Hz | D4 |
| re | 329.63 Hz | E4 |
| mi | 369.99 Hz | F♯4 (approximate — variable in practice) |
| sol | 440.00 Hz | A4 |
| la | 493.88 Hz | B4 |

For each frame we record the nearest degree and the **deviation in cents**:
`cents(f) = 1200 · log2(f / f_ref)`. Zero cents is exactly the reference; ±50 is a
quarter-tone; ±100 a semitone. The systematic, non-zero deviation of Somali
performance from 12-TET **is the quantity of scholarly interest** — the
microtonality no Western-trained model has measured for this tradition.

Two methodological notes, stated plainly (they are also in the paper):

1. **The reference frequencies are calibration targets, not assertions.** They are
   initial estimates to be refined empirically against a master performer's
   intonation — the method *measures* the tradition's tuning rather than imposing
   a Western one.
2. **The mapping code below is verbatim** from the platform's unit-tested module
   `apps/ai-service/services/scale.py`, so every number in this notebook is
   reproducible bit-for-bit by the production service and CI-verified tests.

CREPE settings match the platform: `model_capacity="full"`, `viterbi=True` (smooth
melody contour), `step_size=10` ms; frames below τ = 0.80 confidence are dropped,
and the **voiced fraction** (share of frames passing the gate) is reported per
track. Output: `pitch_data/track_XXXX_pitch.json`, checkpointed per track.


In [ ]:
%pip -q install crepe tensorflow

import crepe
import numpy as np
import soundfile as sf
from tqdm.auto import tqdm

# ── BEGIN verbatim from apps/ai-service/services/scale.py ────────────────────
# Somali pentatonic scale — approximate Hz (D root, common oud tuning).
SOMALI_SCALE_HZ: dict[str, float] = {
    "do": 293.66,  # D4
    "re": 329.63,  # E4
    "mi": 369.99,  # F#4 (approximate — slightly variable in practice)
    "sol": 440.00,  # A4
    "la": 493.88,  # B4
}


def hz_to_somali_note(hz: float) -> tuple[str, float]:
    """Map a frequency to the nearest Somali scale degree.

    Returns ``(note_name, cents_deviation)`` where ``cents_deviation`` reveals
    microtonality:
      * 0 cents      = exactly on the equal-tempered pitch,
      * ±50 cents    = a quarter tone away,
      * ±100 cents   = one semitone away.

    Raises ``ValueError`` for a non-positive frequency (cents are undefined there).
    """
    if hz <= 0:
        raise ValueError("frequency must be positive")

    note_name, target_hz = min(
        SOMALI_SCALE_HZ.items(),
        key=lambda item: abs(hz - item[1]),
    )
    cents_deviation = 1200.0 * math.log2(hz / target_hz)
    return note_name, round(cents_deviation, 2)


def map_pitch_frames(
    frames: list,
    confidence_threshold: float,
) -> list:
    """Turn raw ``(time_sec, frequency_hz, confidence)`` frames into scale-mapped
    pitch points, dropping low-confidence and non-positive frames.

    Kept dependency-free so it composes with CREPE output (arrays converted to a
    list of tuples upstream) while remaining unit-testable on its own.
    """
    points = []
    for time_sec, hz, confidence in frames:
        if confidence < confidence_threshold or hz <= 0:
            continue
        note, deviation = hz_to_somali_note(hz)
        points.append(
            {
                "time_sec": round(time_sec, 3),
                "frequency_hz": round(hz, 2),
                "confidence": round(confidence, 3),
                "note_label": note,
                "cents_deviation": deviation,
            }
        )
    return points
# ── END verbatim from apps/ai-service/services/scale.py ──────────────────────


def analyse_track(no_vocals: Path) -> dict:
    audio, sr = sf.read(str(no_vocals), always_2d=True)
    mono = audio.mean(axis=1).astype(np.float32)
    times, freqs, confs, _ = crepe.predict(
        mono, sr,
        model_capacity="full",   # research data, not a live meter
        viterbi=True,            # smooth melody contour
        step_size=CREPE_STEP_MS, # 10 ms frames (§10)
        verbose=0,
    )
    frames = [
        (float(t), float(hz), float(conf))
        # Parallel arrays from CREPE; strict zip fails loudly on length drift.
        for t, hz, conf in zip(times, freqs, confs, strict=True)
    ]
    points = map_pitch_frames(frames, CONFIDENCE_THRESHOLD)

    note_counts: dict[str, int] = {}
    per_degree: dict[str, list] = {}
    for p in points:
        note_counts[p["note_label"]] = note_counts.get(p["note_label"], 0) + 1
        per_degree.setdefault(p["note_label"], []).append(p["cents_deviation"])
    dominant = sorted(note_counts, key=note_counts.get, reverse=True)

    return {
        "source": no_vocals.parent.name,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "n_frames_total": len(frames),
        "n_frames_voiced": len(points),
        "voiced_fraction": round(len(points) / max(1, len(frames)), 4),
        "dominant_notes": dominant,
        "per_degree_stats": {
            note: {
                "count": len(devs),
                "mean_cents": round(float(np.mean(devs)), 2),
                "std_cents": round(float(np.std(devs)), 2),
            }
            for note, devs in per_degree.items()
        },
        "points": points,
    }


track_dirs = sorted(SEPARATED_DIR.glob("track_*"))
errors = []
for track_dir in tqdm(track_dirs, desc="CREPE + scale map"):
    out = PITCH_DIR / f"{track_dir.name}_pitch.json"
    no_vocals = track_dir / "no_vocals.wav"
    if out.exists() or not no_vocals.exists():
        continue
    try:
        save_json(out, analyse_track(no_vocals))
    except Exception as exc:
        errors.append({"track": track_dir.name, "error": str(exc)})

if errors:
    import pandas as pd
    pd.DataFrame(errors).to_csv(REPORTS_DIR / "errors_step7.csv", index=False)
print(f"Pitch analyses on Drive: {len(list(PITCH_DIR.glob('*_pitch.json')))} "
      f"({len(errors)} failures logged)")


## STEP 8 — Assemble the dataset

Everything joins on `track_id`: filename metadata (Step 2), quality audit
(Step 3), transcripts (Step 6), pitch analysis (Step 7). One JSON record per
track, plus a flat CSV for spreadsheet users.

Two fields deserve honesty rather than cleverness:

- **`quality_score`** is a transparent 0–100 composite: the SNR estimate scaled
  into [0, 100] (20 dB → 50, 40 dB → 100) minus a 10-point clipping penalty when
  peak ≥ −0.1 dBFS. The formula is stated so reviewers can recompute it.
- **`genre_predicted` / `era_estimated`** are taken from the archive CSV when it
  provides them, and otherwise left `null` with `needs_expert_review: true`.
  A pipeline must not invent cultural labels — genre and era assignments belong
  to Somali music scholars and the elders who carry this tradition; false labels
  would be worse than missing ones (provenance is the dataset's value).


In [ ]:
import pandas as pd

metadata = pd.read_csv(METADATA_RAW_CSV)
quality = pd.read_csv(QUALITY_CSV)
transcripts = load_json(TRANSCRIPTS_JSON, {})

TRACK_ID_RE = re.compile(r"^track_(\d+)_", re.IGNORECASE)


def quality_score(snr_db, peak_db) -> float:
    if snr_db is None or (isinstance(snr_db, float) and math.isnan(snr_db)):
        return 0.0
    score = max(0.0, min(100.0, (snr_db / 40.0) * 100.0))
    if peak_db is not None and not math.isnan(peak_db) and peak_db >= -0.1:
        score -= 10.0  # clipping penalty
    return round(max(0.0, score), 1)


records = []
for _, row in metadata.iterrows():
    m = TRACK_ID_RE.match(str(row["filename"]))
    track_key = f"track_{int(row['track_id']):04d}" if pd.notna(row["track_id"]) else None
    q = quality[quality["filename"] == row["filename"]]
    q = q.iloc[0] if len(q) else None
    t = transcripts.get(track_key, {}) if track_key else {}
    pitch_file = PITCH_DIR / f"{track_key}_pitch.json" if track_key else None
    pitch = load_json(pitch_file, {}) if pitch_file and pitch_file.exists() else {}

    records.append(
        {
            "track_id": int(row["track_id"]) if pd.notna(row["track_id"]) else None,
            "filename": row["filename"],
            "title": row["title"] if pd.notna(row["title"]) else None,
            "artists": row["artists"] if pd.notna(row["artists"]) else None,
            "duration_sec": float(row["duration_sec"]) if pd.notna(row["duration_sec"]) else None,
            "quality_score": quality_score(
                q["snr_db"] if q is not None else None,
                q["peak_db"] if q is not None else None,
            ),
            "transcript_somali": t.get("somali", {}).get("text"),
            "transcript_english": t.get("english", {}).get("text"),
            "is_singing": t.get("is_singing"),
            "transcript_confidence": t.get("somali", {}).get("mean_confidence"),
            "dominant_notes": pitch.get("dominant_notes"),
            "voiced_fraction": pitch.get("voiced_fraction"),
            "pitch_data_file": f"pitch_data/{track_key}_pitch.json"
                               if pitch else None,
            # Honest labels: never invented by the pipeline (see markdown above).
            "genre_predicted": None,
            "era_estimated": None,
            "needs_expert_review": True,
            "metadata_source": row["metadata_source"],
        }
    )

save_json(DATASET_JSON, records)
flat = pd.DataFrame(records)
flat["dominant_notes"] = flat["dominant_notes"].apply(
    lambda v: " ".join(v) if isinstance(v, list) else None
)
flat.to_csv(DATASET_CSV, index=False)

complete = sum(1 for r in records if r["transcript_somali"] and r["pitch_data_file"])
print(f"Dataset v1: {len(records)} records "
      f"({complete} with transcript + pitch — fully processed)")
print(f"  {DATASET_JSON}")
print(f"  {DATASET_CSV}")


## STEP 9 — Dataset statistics and the paper's Figure 1

The summary below feeds the ⟨placeholders⟩ in `docs/ISMIR_DRAFT.md` directly:
corpus size and duration, quality distribution, the dominant-degree histogram,
the per-degree cents-deviation distributions (**Figure 1**), and a first
transcript vocabulary. Everything is written to Drive
(`dataset_summary.txt`, `figure1_cents_histograms.png` at 300 dpi).

*Figure design note:* the five degrees are **facets, not series** — a single hue
across all panels, identity carried by panel titles; the only reference marks are
a neutral line at 0 cents (the 12-TET reference) and dotted quarter-tone guides
at ±50 cents. What the reader should see instantly is *where each distribution
sits relative to zero* — that offset is the microtonality finding.


In [ ]:
import re as _re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

records = load_json(DATASET_JSON, [])
quality = pd.read_csv(QUALITY_CSV)

# ── Aggregate pitch data across the corpus ───────────────────────────────────
DEGREES = ["do", "re", "mi", "sol", "la"]
per_degree: dict[str, list] = {d: [] for d in DEGREES}
note_counter: Counter = Counter()
n_frames = 0
for pf in sorted(PITCH_DIR.glob("*_pitch.json")):
    data = load_json(pf, {})
    for p in data.get("points", []):
        per_degree.setdefault(p["note_label"], []).append(p["cents_deviation"])
        note_counter[p["note_label"]] += 1
        n_frames += 1

all_devs = np.array([d for devs in per_degree.values() for d in devs]) if n_frames else np.array([])

# ── Figure 1: per-degree cents histograms (single hue; facets, not series) ───
HUE = "#4477AA"          # one series-hue; identity lives in panel titles
INK = "#333333"
GRID = "#DDDDDD"

fig, axes = plt.subplots(1, 5, figsize=(15, 3.2), sharey=True, sharex=True)
for ax, degree in zip(axes, DEGREES, strict=True):
    devs = per_degree.get(degree, [])
    ax.axvline(0, color="#888888", lw=1, ls="--", zorder=1)      # 12-TET reference
    ax.axvline(-50, color=GRID, lw=0.8, ls=":", zorder=1)        # quarter-tone
    ax.axvline(50, color=GRID, lw=0.8, ls=":", zorder=1)
    if devs:
        ax.hist(devs, bins=40, range=(-100, 100), color=HUE, edgecolor="white",
                linewidth=0.3, zorder=2)
        mu = float(np.mean(devs))
        ax.set_title(f"{degree}   μ = {mu:+.1f}¢", color=INK, fontsize=11)
    else:
        ax.set_title(f"{degree}   (no frames)", color=INK, fontsize=11)
    ax.set_xlabel("cents from 12-TET", color=INK, fontsize=9)
    ax.tick_params(colors=INK, labelsize=8)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(axis="y", color=GRID, lw=0.5)
    ax.set_axisbelow(True)
axes[0].set_ylabel("frames", color=INK, fontsize=9)
fig.suptitle("Deviation from equal temperament by Somali scale degree",
             color=INK, fontsize=13)
fig.tight_layout()
fig.savefig(FIGURE1_PNG, dpi=300, bbox_inches="tight")
plt.show()

# ── Vocabulary from the Somali transcripts ───────────────────────────────────
word_counter: Counter = Counter()
for r in records:
    text = r.get("transcript_somali") or ""
    if r.get("is_singing"):
        continue  # advisory transcripts stay out of the vocabulary count
    word_counter.update(w for w in _re.findall(r"[A-Za-z']+", text.lower()) if len(w) > 2)

# ── Summary report ───────────────────────────────────────────────────────────
total_sec = sum(r["duration_sec"] or 0 for r in records)
hours = total_sec / 3600
lines = [
    "SOMALI MUSIC DATASET v1 — SUMMARY",
    "=" * 50,
    f"Recordings:            {len(records)}",
    f"Total duration:        {hours:.1f} hours ({total_sec:,.0f} s)",
    f"Fully processed:       "
    f"{sum(1 for r in records if r['transcript_somali'] and r['pitch_data_file'])}",
    f"Flagged is_singing:    {sum(1 for r in records if r.get('is_singing'))}",
    "",
    "Quality score distribution:",
    f"  mean {np.mean([r['quality_score'] for r in records]):.1f} / "
    f"median {np.median([r['quality_score'] for r in records]):.1f}",
    f"  below SNR gate ({SNR_FLAG_DB:.0f} dB): "
    f"{int(quality['flagged_low_snr'].sum())} files",
    "",
    f"Pitch frames retained (τ = {CONFIDENCE_THRESHOLD}): {n_frames:,}",
]
if n_frames:
    lines += [
        f"Mean |deviation| from 12-TET: {np.mean(np.abs(all_devs)):.1f} cents "
        f"(σ = {np.std(all_devs):.1f})",
        "Scale-degree distribution (Somali pentatonic):",
    ]
    for note, count in note_counter.most_common():
        mu = np.mean(per_degree[note])
        lines.append(f"  {note:>4}: {count:>9,} frames   μ = {mu:+6.1f}¢")
lines += ["", "Top transcript vocabulary (non-singing tracks):",
          "  " + ", ".join(w for w, _ in word_counter.most_common(30))]

report = "\n".join(lines)
SUMMARY_TXT.write_text(report)
print(report)
print(f"\nSaved: {SUMMARY_TXT}")
print(f"Saved: {FIGURE1_PNG} (Figure 1, 300 dpi)")


## What happens next

This notebook's outputs flow into the rest of the platform:

1. **Platform ingest** — the dataset records map onto the archive's
   `PublicRecording` schema; audio uploads via presigned R2 URLs and the AI
   fields (`transcriptSomali`, `pitchData`, `dominantNotes`, `voicedFraction`)
   arrive through the same internal callback the production pipeline uses.
2. **MERT embeddings** — the production service computes MERT-v1-95M embeddings
   into `pgvector` for similarity search; the cleaned stems here are its input.
3. **The ISMIR paper** — Figure 1 and the summary statistics above fill the
   ⟨placeholders⟩ in `docs/ISMIR_DRAFT.md`. The per-degree deviation
   distributions are the headline result.
4. **Expert review** — every record carries `needs_expert_review: true` until
   Somali music scholars confirm titles, artists, genre, and era. The reference
   frequencies in `SOMALI_SCALE_HZ` are then re-calibrated against master
   recordings and the cents distributions recomputed — the method measures the
   tradition; it does not impose on it.

**Contact:** Khalid Ibrahim · Somali Music AI Preservation Platform
Repository: `somali-music-archive` · Paper draft: `docs/ISMIR_DRAFT.md`
